# Telegram → trading-library book export

Run this notebook in Google Colab (colab.research.google.com → File → Upload notebook, or open directly from GitHub).

It downloads document attachments (PDF/EPUB/MOBI/AZW3/DJVU/TXT/MD) from a Telegram channel into `books/`, then zips them for download.

**Credentials are entered interactively below and never written to this file or committed anywhere.**

In [ ]:
!pip install -q telethon

In [ ]:
import getpass

TG_API_ID = input("Telegram API ID: ").strip()
TG_API_HASH = getpass.getpass("Telegram API hash: ").strip()
TG_CHANNEL = input("Channel username (e.g. mychannel) or numeric chat ID (e.g. -1001599776120): ").strip()

In [ ]:
import asyncio
import re
from pathlib import Path

from telethon import TelegramClient
from telethon.tl.types import DocumentAttributeFilename

BOOKS_DIR = Path("books")
SESSION_NAME = "telegram_export"
DOCUMENT_EXTENSIONS = {".pdf", ".epub", ".mobi", ".azw3", ".djvu", ".txt", ".md"}


def sanitize_filename(name: str) -> str:
    name = re.sub(r"[^\w\s.\-]", "", name).strip()
    return re.sub(r"\s+", "_", name)


def get_filename(message):
    if not message.document:
        return None
    for attr in message.document.attributes:
        if isinstance(attr, DocumentAttributeFilename):
            return attr.file_name
    return None


async def export():
    BOOKS_DIR.mkdir(exist_ok=True)
    client = TelegramClient(SESSION_NAME, TG_API_ID, TG_API_HASH)
    await client.start()

    entity = await client.get_entity(int(TG_CHANNEL) if re.fullmatch(r"-?\d+", TG_CHANNEL) else TG_CHANNEL)

    downloaded = 0
    async for message in client.iter_messages(entity):
        filename = get_filename(message)
        if not filename:
            continue
        ext = Path(filename).suffix.lower()
        if ext not in DOCUMENT_EXTENSIONS:
            continue

        dest = BOOKS_DIR / sanitize_filename(filename)
        if dest.exists():
            continue

        print(f"Downloading {filename} -> {dest}")
        await client.download_media(message, file=str(dest))
        downloaded += 1

    print(f"Done. Downloaded {downloaded} new file(s) into {BOOKS_DIR}/")
    await client.disconnect()


await export()

## Download the results

Zips everything in `books/` and triggers a browser download (Colab only). If not on Colab, just grab the files from the `books/` folder directly.

In [ ]:
!zip -rq books.zip books/

try:
    from google.colab import files
    files.download("books.zip")
except ImportError:
    print("Not running in Colab — books.zip created in the working directory, download manually.")